# MambaGuard — GPU study on Kaggle

Runs every benchmark × seed, the ablations, the smoothing certificates and the GPU latency, and writes one zip
(`/kaggle/working/results.zip`) that you download from the **Output** tab.

**Before running:** Settings (right panel) → *Accelerator* = **GPU T4 x2**, *Internet* = **On**
(Internet needs a phone-verified Kaggle account). Keep the notebook **Private**.

Run the cells top to bottom. Cell 7 is a ~5-minute smoke test; only start cell 8 once it finishes without errors.
For the full study, use **Save Version → Save & Run All (Commit)** so it keeps running after you close the browser (12 h limit).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -- set Accelerator to GPU T4 x2")
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print(f"compute capability {major}.{minor}; bf16 supported: {torch.cuda.is_bf16_supported()}")
    if major < 7:
        print("WARNING: this GPU (e.g. P100) may not be supported by Kaggle's current PyTorch build. Switch to T4 x2.")

## 2. Get the code

Option A (default): clone the public GitHub branch (needs Internet).
Option B: upload `mambaguard-kaggle.zip` as a Kaggle *Dataset* and attach it with **+ Add Input**; this cell finds it
automatically under `/kaggle/input` and uses it instead of GitHub.

In [ ]:
import os, glob, subprocess, shutil
REPO_URL = "https://github.com/rogerpanel/CV.git"
BRANCH   = "claude/implement-mambaguard-detection-zCgu6"
CODE = None
for init in glob.glob("/kaggle/input/**/mambaguard/__init__.py", recursive=True):
    src = os.path.dirname(os.path.dirname(init))
    CODE = "/kaggle/working/mambaguard_code"          # copy: /kaggle/input is read-only
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(src, CODE, ignore=shutil.ignore_patterns("__pycache__"))
    print("using uploaded dataset:", src)
    break
if CODE is None:
    shutil.rmtree("/tmp/mg_repo", ignore_errors=True)
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, "--filter=blob:none",
                    "--sparse", REPO_URL, "/tmp/mg_repo"], check=True)
    subprocess.run(["git", "-C", "/tmp/mg_repo", "sparse-checkout", "set", "MambaGuard"], check=True)
    CODE = "/tmp/mg_repo/MambaGuard"
    print("cloned", BRANCH, "commit", subprocess.check_output(["git", "-C", "/tmp/mg_repo", "rev-parse", "--short", "HEAD"], text=True).strip())
os.chdir(CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, CODE)
print("code at", os.getcwd()); print(sorted(os.listdir(".")))

## 3. Install the two extra packages (torch is already installed — do not reinstall it)

In [ ]:
!pip install -q -r kaggle/requirements-kaggle.txt
import einops, sentence_transformers; print("einops", einops.__version__, "| sentence-transformers", sentence_transformers.__version__)

## 4. Download the benchmarks at their pinned commits

Saved to `/tmp/mambaguard_data` (not `/kaggle/working`, so they are not part of the output: MCPTox and
MCP-SafetyBench have no licence and must not be redistributed). AgentDojo is the largest (~0.5 GB, 1–3 min).
The A2A sandbox is generated, no download.

In [ ]:
DATA = "/tmp/mambaguard_data"
!bash kaggle/setup_benchmarks.sh {DATA} mcptox mcp_safetybench agentdojo injecagent

## 5. Check the loaders (episode counts should match `reproducibility/PROTOCOL_BENCHMARKS.md`)

In [ ]:
from mambaguard.data.protocol_traces import EPISODE_LOADERS
for name, folder in [("mcptox", "MCPTox-Benchmark"), ("mcp_safetybench", "MCPSafety"),
                     ("agentdojo", "agentdojo"), ("injecagent", "InjecAgent"), ("a2a_sandbox", "builtin")]:
    root = folder if folder == "builtin" else f"{DATA}/{folder}"
    eps = EPISODE_LOADERS[name](root, "all")
    att = sum(e.is_attack for e in eps)
    print(f"{name:16s} episodes={len(eps):5d}  attack={att:5d}  benign={len(eps) - att:5d}")

## 6. Pre-download the MiniLM sentence encoder (~90 MB)

In [ ]:
from sentence_transformers import SentenceTransformer
m = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
print("embedding dim", m.get_sentence_embedding_dimension())
del m

## 7. Smoke test (~5 min): tiny data, 2 epochs, every stage. Must finish with no `ERROR` lines.

In [ ]:
!python kaggle/run_kaggle_study.py --smoke --data-root {DATA} --out /kaggle/working/results
!python scripts/summarise_kaggle_results.py /kaggle/working/results_smoke.zip | head -40

## 8. Full study

Defaults: 5 benchmarks × 3 seeds, 5 ablations × 2 seeds on A2A + MCPTox, certificates (σ = 0.05/0.1/0.25,
100 windows, n = 500) and latency. It stops starting new runs after `--time-budget-h` hours and zips what it has.

If it runs out of time: commit a new version with this notebook's previous output attached (**+ Add Input →
Notebook Output**) and set `RESUME` to that folder — finished runs are copied and skipped.

To shorten it: `--seeds 0 1`, `--skip-ablations`, `--epoch-scale 0.5`, or `--benchmarks a2a_sandbox mcptox`.

In [ ]:
RESUME = ""   # e.g. "/kaggle/input/mambaguard-gpu-study/results"  (output of an earlier version)
EXTRA  = ""   # e.g. "--seeds 0 1 --skip-ablations"
resume_flag = f"--resume-from {RESUME}" if RESUME else ""
!python kaggle/run_kaggle_study.py --data-root {DATA} --out /kaggle/working/results --time-budget-h 11 {resume_flag} {EXTRA}

## 9. Summary and download

In [ ]:
!python scripts/summarise_kaggle_results.py /kaggle/working/results.zip --latex
from IPython.display import FileLink, display
!ls -lh /kaggle/working/*.zip
os.chdir("/kaggle/working"); display(FileLink("results.zip")); os.chdir(CODE)
# or: Output tab (right panel, or the version page after a commit) -> results.zip -> Download